# 3. Transform – bronze ➜ silver
| Entrada (bronze) | Salida (silver) |
|---|---|
| `car_features` | `car_features_clean` – una fila por marca/modelo |
| `car_sales` + `dealers` + `car_features_clean` | `car_sales_enriched` – ventas limpias y enriquecidas |

**Técnicas aplicadas:** limpieza de nulos y duplicados, corrección de texto con `regexp_replace`, funciones de ventana (`dense_rank`), agregaciones (`mode`, `avg`), UDF, `when/otherwise`, tabla de equivalencias, joins con `broadcast` y columnas derivadas.

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalogo", "vehiculos")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_sales = spark.table(f"{catalogo}.{esquema_source}.car_sales")
df_features = spark.table(f"{catalogo}.{esquema_source}.car_features")
df_dealers = spark.table(f"{catalogo}.{esquema_source}.dealers")

## Parte 1 – Ficha técnica (`car_features_clean`)
### 1.1 Limpieza: duplicados, nulos y llaves normalizadas

In [0]:
df_features_clean = df_features.drop("source_file", "ingestion_date") \
                               .dropDuplicates() \
                               .dropna(subset=["make", "model", "year"]) \
                               .withColumn("make", upper(trim(col("make")))) \
                               .withColumn("model", upper(trim(col("model"))))

print(f"Filas bronze: {df_features.count()} | Filas sin duplicados: {df_features_clean.count()}")

### 1.2 Función de ventana: quedarse con el año más reciente de cada modelo
`dense_rank` sobre `year` descendente, particionado por marca y modelo.

In [0]:
ventana_modelo = Window.partitionBy("make", "model").orderBy(col("year").desc())

df_features_latest = df_features_clean \
    .withColumn("rank_year", dense_rank().over(ventana_modelo)) \
    .filter(col("rank_year") == 1) \
    .drop("rank_year")

### 1.3 Agregación: una fila por marca/modelo
Un mismo modelo tiene varias versiones (trims) en el mismo año: se toma el valor más frecuente (`mode`) para atributos de texto y el promedio para los numéricos.

In [0]:
df_features_agg = df_features_latest.groupBy("make", "model").agg(
    F.max("year").alias("latest_year"),
    F.mode("engine_fuel_type").alias("engine_fuel_type"),
    F.round(F.avg("engine_hp")).cast(IntegerType()).alias("engine_hp"),
    F.max("engine_cylinders").alias("engine_cylinders"),
    F.mode("transmission_type").alias("transmission_type"),
    F.mode("driven_wheels").alias("driven_wheels"),
    F.mode("vehicle_size").alias("vehicle_size"),
    F.mode("vehicle_style").alias("vehicle_style"),
    F.round(F.avg("highway_mpg")).cast(IntegerType()).alias("highway_mpg"),
    F.round(F.avg("city_mpg")).cast(IntegerType()).alias("city_mpg"),
    F.max("popularity").alias("popularity"),
    F.round(F.avg("msrp"), 2).alias("msrp_avg"),
    F.max(coalesce(col("market_category").contains("Luxury"), lit(False))).alias("is_luxury")
).withColumn("ingestion_date", current_timestamp())

display(df_features_agg.orderBy("make", "model"))

In [0]:
columnas = spark.table(f"{catalogo}.{esquema_sink}.car_features_clean").columns
df_features_agg.select(*columnas).write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.car_features_clean")

## Parte 2 – Ventas (`car_sales_enriched`)
### 2.1 Limpieza
- `engine` trae caracteres mal codificados (`DoubleÂ Overhead Camshaft`): se eliminan con `regexp_replace`.
- Nombre de cliente nulo ➜ `Desconocido`.
- Duplicados por `car_id` y precios no válidos.

In [0]:
df_sales_clean = df_sales.dropna(how="all") \
    .filter(col("car_id").isNotNull() & col("sale_date").isNotNull()) \
    .filter(col("price") > 0) \
    .dropDuplicates(["car_id"]) \
    .withColumn("engine", trim(regexp_replace(regexp_replace(col("engine"), "[^A-Za-z ]", " "), " +", " "))) \
    .fillna({"customer_name": "Desconocido"}) \
    .withColumn("customer_name", initcap(trim(col("customer_name")))) \
    .withColumn("dealer_no", trim(col("dealer_no")))

display(df_sales_clean.groupBy("engine").count())

### 2.2 Columnas derivadas de fecha, UDF y segmentaciones
- `income_segment` con una **UDF** de Python.
- `price_segment` con `when / otherwise`.

In [0]:
def segmento_ingreso(ingreso):
    if ingreso is None:
        return "Sin dato"
    elif ingreso < 400000:
        return "Bajo"
    elif ingreso < 800000:
        return "Medio"
    elif ingreso < 1500000:
        return "Alto"
    else:
        return "Premium"

segmento_ingreso_udf = F.udf(segmento_ingreso, StringType())

In [0]:
df_sales_derived = df_sales_clean \
    .withColumn("sale_month", month(col("sale_date"))) \
    .withColumn("sale_quarter", quarter(col("sale_date"))) \
    .withColumn("income_segment", segmento_ingreso_udf(col("annual_income"))) \
    .withColumn("price_segment",
                when(col("price") < 15000, "Económico")
                .when(col("price") < 30000, "Medio")
                .when(col("price") < 50000, "Alto")
                .otherwise("Lujo"))

### 2.3 Normalización de marca/modelo para cruzar con la ficha técnica
La fuente de ventas escribe los modelos distinto a la ficha técnica. Se corrigen con:
1. **Reglas con expresiones regulares** (Lexus `LS400` ➜ `LS 400`; BMW `328I` ➜ `3 SERIES`).
2. **Tabla de equivalencias** para casos puntuales (incluye `3-Sep` ➜ `9-3`: Excel convirtió el modelo Saab *9-3* en una fecha).

In [0]:
equivalencias = [
    ("MERCEDES-BENZ", "SLK",            "SLK-CLASS"),
    ("MERCEDES-BENZ", "SLK230",         "SLK-CLASS"),
    ("MERCEDES-BENZ", "CL500",          "CL-CLASS"),
    ("MERCEDES-BENZ", "CLK COUPE",      "CLK-CLASS"),
    ("DODGE",         "RAM PICKUP",     "RAM PICKUP 1500"),
    ("CHRYSLER",      "SEBRING COUPE",  "SEBRING"),
    ("CHRYSLER",      "SEBRING CONV.",  "SEBRING"),
    ("PORSCHE",       "CARRERA COUPE",  "911"),
    ("PORSCHE",       "CARRERA CABRIO", "911"),
    ("PORSCHE",       "BOXTER",         "BOXSTER"),
    ("SAAB",          "3-SEP",          "9-3"),
    ("SAAB",          "5-SEP",          "9-5"),
    ("FORD",          "F-SERIES",       "F-150"),
]
df_equivalencias = spark.createDataFrame(equivalencias, ["make_key", "model_key", "model_equivalente"])

In [0]:
df_sales_keys = df_sales_derived \
    .withColumn("company", when(upper(col("company")) == "MERCEDES-B", "Mercedes-Benz").otherwise(col("company"))) \
    .withColumn("make_key", upper(trim(col("company")))) \
    .withColumn("model_key", regexp_replace(upper(trim(col("model"))), "&", "AND")) \
    .withColumn("model_key",
                when(col("make_key") == "LEXUS", regexp_replace(col("model_key"), "^([A-Z]{2})([0-9])", "$1 $2"))
                .when(col("make_key") == "BMW", regexp_replace(col("model_key"), "^([0-9])[0-9]{2}I$", "$1 SERIES"))
                .otherwise(col("model_key")))

df_sales_keys = df_sales_keys.join(broadcast(df_equivalencias), ["make_key", "model_key"], "left") \
    .withColumn("model_key", coalesce(col("model_equivalente"), col("model_key"))) \
    .drop("model_equivalente")

### 2.4 Joins con sucursales y ficha técnica
- `dealers` es un catálogo pequeño ➜ **broadcast join**.
- `region_consistent`: valida si la región registrada en la venta coincide con la ciudad de la sucursal.
- `price_vs_msrp_pct`: diferencia porcentual entre el precio de venta y el precio de lista.

In [0]:
df_dealers_sel = df_dealers.select("dealer_no", "city", "state_code", "state_name", "latitude", "longitude")

df_features_sel = spark.table(f"{catalogo}.{esquema_sink}.car_features_clean") \
    .select(col("make").alias("make_key"), col("model").alias("model_key"),
            "engine_hp", "vehicle_size", "engine_fuel_type", "msrp_avg")

df_joined = df_sales_keys.alias("s") \
    .join(broadcast(df_dealers_sel).alias("d"), col("s.dealer_no") == col("d.dealer_no"), "left") \
    .drop(col("d.dealer_no")) \
    .join(df_features_sel, ["make_key", "model_key"], "left")

In [0]:
df_enriched = df_joined \
    .withColumn("region_consistent", col("dealer_region") == col("city")) \
    .withColumn("has_features", col("msrp_avg").isNotNull()) \
    .withColumn("price_vs_msrp_pct",
                F.round((col("price") - col("msrp_avg")) / col("msrp_avg") * 100, 2)) \
    .withColumn("ingestion_date", current_timestamp())

### 2.5 Indicadores de calidad

In [0]:
total = df_enriched.count()
con_ficha = df_enriched.filter(col("has_features")).count()
sin_sucursal = df_enriched.filter(col("city").isNull()).count()
region_ok = df_enriched.filter(col("region_consistent")).count()

print(f"Ventas procesadas          : {total}")
print(f"Con ficha técnica          : {con_ficha} ({con_ficha / total:.1%})")
print(f"Sin sucursal en catálogo   : {sin_sucursal}")
print(f"Región consistente         : {region_ok} ({region_ok / total:.1%})")

display(df_enriched.filter(~col("has_features")).groupBy("company").count().orderBy(col("count").desc()))

### 2.6 Escritura en silver

In [0]:
columnas = spark.table(f"{catalogo}.{esquema_sink}.car_sales_enriched").columns
df_enriched.select(*columnas).write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.car_sales_enriched")

In [0]:
display(spark.table(f"{catalogo}.{esquema_sink}.car_sales_enriched").limit(20))